## Co-culture growth-rate QC

Quantifies the GFP-positive fraction of A549-doxCas9 co-cultures over time
(flow cytometry, CytoFLEX) and fits a logistic competition model to estimate
each knockout's growth rate relative to wild type.

**Inputs** (placed under `data/figures/SI/`): a tube-name workbook
(`flow_tubenames_*.xlsx`) and the per-batch eFlow GFP-gate exports
(`gfp_pos_*.xlsx`). **Outputs** are written to `output/SI/`.

In [ ]:
# parameters
flow_sheet_name = ['growthQC_4_5', 'gQC_CER2_7-9-10', 'growthQC_2_3_8_11_12', 'growthQC_17_20_22_23', 'growthQC_13_15_16_21', 'growthQC_CLE4_1', 'growthQC_CER1', 'growthQC_29_31']
eflowQ_excel_name = ['gfp_pos_CER2_2.xlsx', 'gfp_pos_CER2_3.xlsx', 'gfp_pos_CER2.xlsx', 'gfp_pos_CER3.xlsx', 'gfp_pos_CER3_2.xlsx', 'gfp_pos_CER4.xlsx', 'gfp_pos_CER1.xlsx', 'gfp_pos_CER4_2.xlsx']
flow_tubenames_file = 'flow_tubenames_20250813.xlsx'

gate_name = 'gfp'
control_tube = 'A549-doxCas9-C8'

# logistic growth-model constants
tau_wt = 1.25            # wild-type doubling time (days); reference for relative growth
REF_DAY = 4              # day at which relative growth (relg4) is reported
RELG4_THRESHOLD = 0.72   # relative-growth cutoff drawn on the histogram

In [ ]:
%matplotlib inline
from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (10, 6)
mpl.rcParams['pdf.fonttype'] = 42  # keep text editable in exported PDFs

In [ ]:
# input/output paths -- data/ is a symlink to the central HPC analysis folder
FIGURE_DATA = Path("../../data/figures/SI/CLE_QC")
OUT_DIR = Path("../../output/SI")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# import data
flowsheet = pd.DataFrame()
for sheet in flow_sheet_name:
    df = pd.read_excel(FIGURE_DATA / flow_tubenames_file, sheet_name=sheet)
    flowsheet = pd.concat([flowsheet, df])

eflowQ_excel = pd.DataFrame()
for name in eflowQ_excel_name:
    df = pd.read_excel(FIGURE_DATA / name)
    eflowQ_excel = pd.concat([eflowQ_excel, df])

days = np.sort(flowsheet['day'].unique())
cellline = sorted(flowsheet['cell_line'].unique(), reverse=True)

## Co-culture assay

In [ ]:
# Build per-(cell line, day) GFP-positive fractions from the eFlow exports
experiment_type = 'co_culture'

data = []
for day in days:
    for cell in cellline:
        try:
            experiment_type_temp = experiment_type
            if cell == control_tube:
                experiment_type_temp = 'indv_culture'

            # locate the directory + fcs file for this cell line / day
            mask = (
                (flowsheet['cell_line'] == cell) &
                (flowsheet['day'] == day) &
                (flowsheet['experiment'] == experiment_type_temp)
            )
            dir_val = flowsheet.loc[mask, 'directory'].iloc[0]
            fcs_val = flowsheet.loc[mask, 'fcs'].iloc[0]

            # find the matching GFP-positive fraction in the eFlow export
            pattern = re.escape(dir_val) + r'.*' + re.escape(fcs_val)
            tube_mask = eflowQ_excel['Unnamed: 0'].str.contains(pattern, regex=True)
            gate_col = '% of parent in: ' + chr(10) + gate_name + ' (selected)'
            gfp_value = eflowQ_excel.loc[tube_mask, gate_col].iloc[0]

            data.append([cell, day, gfp_value])
            print(f"Day: {day}, Cell line: {cell}, GFP-pos fraction: {gfp_value}")
        except IndexError:
            print(f"IndexError: No GFP-pos fraction available for Day: {day}, Cell line: {cell}")

df_cocul = pd.DataFrame(data, columns=['cellline', 'days', 'PosFrac'])

In [ ]:
# Logistic competition model: GFP-positive fraction over time, set by the
# knockout's doubling time tau relative to wild type (tau_wt).
def logistic_growth(t, x0, tau):
    return 1 / (1 + x0 * 2**((1/tau_wt - 1/tau) * t))

num_cell_lines = len(df_cocul['cellline'].unique())
colors = plt.get_cmap('tab10', num_cell_lines)
n_grid = int(np.ceil(np.sqrt(num_cell_lines)))

tau_fit_res = []
cellline_name = []
plt.figure(figsize=(15, 15))
for i, cell_line in enumerate(df_cocul['cellline'].unique()):
    if cell_line == control_tube:
        continue
    cell_df = df_cocul[df_cocul['cellline'] == cell_line]
    plt.subplot(n_grid, n_grid, i + 1)
    plt.plot(cell_df['days'], cell_df['PosFrac'], 'o', color=colors(i), label=cell_line)

    try:
        popt, pcov = curve_fit(logistic_growth, cell_df['days'], cell_df['PosFrac'], p0=[1, tau_wt])
        x0_fit, tau_fit = popt
        x_fit = np.linspace(cell_df['days'].min(), cell_df['days'].max(), 100)
        y_fit = logistic_growth(x_fit, x0_fit, tau_fit)
        plt.plot(x_fit, y_fit, '-', color=colors(i),
                 label=f'Fit (tauR={tau_fit/tau_wt:.2f}, relative growth at d{REF_DAY}={2**((1/tau_fit - 1/tau_wt)*REF_DAY):.2f})')
        tau_fit_res.append(tau_fit)
        cellline_name.append(cell_line)
    except RuntimeError:
        print(f"Error fitting curve for {cell_line}")

    plt.xlim(-1, 12)
    plt.ylim(0, 1)
    plt.xlabel('Days')
    plt.ylabel('Fraction')
    plt.title(cell_line)
plt.tight_layout()
plt.savefig(OUT_DIR / 'PosFrac_lineplot_fitted.png', dpi=300)
plt.show()

df_tau = pd.DataFrame(list(zip(cellline_name, tau_fit_res)), columns=['cellline', 'tau'])
df_tau['relg4'] = 2**((1/df_tau['tau'] - 1/tau_wt) * REF_DAY)
df_tau['tauR'] = df_tau['tau'] / tau_wt
df_tau

In [ ]:
plt.figure(figsize=(4, 4))
sns.histplot(data=df_tau, x='relg4', kde=True, binrange=(0, 1.2), binwidth=0.06)
plt.axvline(x=RELG4_THRESHOLD, color='r', linestyle='--')
plt.xlabel(f'Relative Growth at day{REF_DAY}')
plt.ylabel('Number of cell lines')
plt.tight_layout()
plt.savefig(OUT_DIR / 'Histogram_Relg4.pdf')
plt.savefig(OUT_DIR / 'Histogram_Relg4.svg')
plt.savefig(OUT_DIR / 'Histogram_Relg4.png', dpi=300)
plt.show()

In [ ]:
# Representative subset of knockouts for the main-figure panel
cell_line_to_visualize = ["RPL36", "SEC23A", "NPM1", "LAMP1"]
colors = plt.get_cmap('tab10', len(cell_line_to_visualize))

plt.figure(figsize=(3.5, 3.5))
for i, cell_line in enumerate(cell_line_to_visualize):
    cell_df = df_cocul[df_cocul['cellline'] == f"{control_tube} + {cell_line}"]
    plt.plot(cell_df['days'], cell_df['PosFrac'], 'o', color=colors(i), label=cell_line)

    try:
        popt, pcov = curve_fit(logistic_growth, cell_df['days'], cell_df['PosFrac'], p0=[1, tau_wt])
        x0_fit, tau_fit = popt
        x_fit = np.linspace(cell_df['days'].min(), cell_df['days'].max(), 100)
        y_fit = logistic_growth(x_fit, x0_fit, tau_fit)
        plt.plot(x_fit, y_fit, '-', color=colors(i))
    except RuntimeError:
        print(f"Error fitting curve for {cell_line}")

    plt.xlim(-1, 11)
    plt.ylim(0, 1)
    plt.xlabel('Days')
    plt.ylabel('Fraction')
    plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(OUT_DIR / 'PosFrac_lineplot_fitted_representative.png', dpi=300)
plt.savefig(OUT_DIR / 'PosFrac_lineplot_fitted_representative.svg')
plt.savefig(OUT_DIR / 'PosFrac_lineplot_fitted_representative.pdf')
plt.show()